## ETL

In [14]:
import pandas as pd
import sqlite3

# ============ Extract ================
# Read raw data from CSV files
customers = pd.read_csv("customers.csv",sep=",")
#print(customers.columns)
transactions = pd.read_csv("transactions.csv",sep=",")
#print(transactions.columns)
#print(customers.head())
#print(customers.columns.tolist())

#print(transactions.head())
#print(transactions.columns.tolist())

# Fix columns by splitting the single combined column
customers = customers['CustomerID,Name,Region'].str.split(',', expand=True)
customers.columns = ['CustomerID','Name','Region']
transactions = transactions['TransID,CustomerID,Date,Amount,Status'].str.split(',', expand=True)
transactions.columns = ['TransID', 'CustomerID', 'Date', 'Amount', 'Status']


# ============ Transform ==============
# 1. Remove cancelled transactions
transactions = transactions[transactions['Status'] == 'Completed']

# 2. Handle missing values (fill NaN names with 'Unknown')
#customers['Name'] = customers['Name'].fillna('Unknown')
customers['Name'] = customers['Name'].replace(r'^\s*$', 'Unknown', regex=True)   # a string that contains nothing, or only spaces/tabs

# 3. Merge data for analysis
data = pd.merge(transactions, customers, on='CustomerID', how='inner')

# 4. Convert Date string to datetime type
data['Date'] = pd.to_datetime(data['Date'])

# 5. Optional: Create derived column (e.g., 'Year')
data['Year'] = data['Date'].dt.year

# ============ Load ===================
# Store the result in a SQLite database
conn = sqlite3.connect(r"C:\Users\GURUVAMMAL\warehouse.db")
data.to_sql("FactCustomerPurchases", conn, if_exists="replace", index=False)
conn.close()

print("ETL complete. Data loaded into FactCustomerPurchases table.")


ETL complete. Data loaded into FactCustomerPurchases table.


###
.db is binary, not UTF-8 text,so editors complain “not UTF-8 encoded”.

You can’t edit or view .db files as plain text. 
You need to open them with a database tool


In [15]:
import sqlite3
import pandas as pd

# connect to your database
conn = sqlite3.connect(r"C:\Users\GURUVAMMAL\warehouse.db")

# list tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

# read one table (example: Customers)
df = pd.read_sql("SELECT * FROM FactCustomerPurchases;", conn)
print(df.head())

conn.close()


                    name
0  FactCustomerPurchases
  TransID CustomerID                 Date Amount     Status     Name Region  \
0    T001       C101  2024-05-01 00:00:00   1000  Completed    Alice  North   
1    T003       C102  2024-05-22 00:00:00   1250  Completed      Bob  South   
2    T004       C103  2024-07-10 00:00:00    300  Completed  Unknown   East   

   Year  
0  2024  
1  2024  
2  2024  
